In [ ]:
import os
import sys

import collections
import xarray as xr
xr.set_options(keep_attrs=True)
import netCDF4 as nc
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
import metpy.calc as mp
from metpy.units import units
from scipy.stats import ttest_ind, ttest_rel
from datetime import datetime

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

In [ ]:
def windSpd(u,v):
   windSpd=np.sqrt(u**2 + v**2)
   return windSpd

def froude(N,h,Uspd):
   fr=(N*h)/Uspd
   return fr

def froude_from_upstream_vars(topo,u,v,ts,ta,z,
                              p2wind=800,
                              p2n=600,
                              oro_lat_bounds=None,
                              oro_lon_bounds=None,
                              ups_lat_bounds=None,
                              ups_lon_bounds=None):
    ## Clip topo data to corresponding lat/lon bounds
    print('Clipping topography...')
    tymin=oro_lat_bounds[0] ; tymax=oro_lat_bounds[1]
    txmin=oro_lon_bounds[0] ; txmax=oro_lon_bounds[1]
    h=topo.sel(lat=slice(tymin,tymax), lon=slice(txmin,txmax))
    
    ## Define pressure level thresholds for wind and N vars
    surface_plev=1000
    wind_p2lev=p2wind
    wind_p2idx=int(np.where(u.lev==wind_p2lev)[0])
    n_p2lev=p2n
    n_p2idx=int(np.where(u.lev==n_p2lev)[0])

    ## Clip upstream climate data and calculate zonal regional means
    print('Calculating Zonal Mean Upstream Uspd...')
    ymin=ups_lat_bounds[0] ; ymax=ups_lat_bounds[1]
    xmin=ups_lon_bounds[0] ; xmax=ups_lon_bounds[1]
    # Wind Speed
    uwnd=u[:,0:wind_p2idx,:,:].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax))
    vwnd=v[:,0:wind_p2idx,:,:].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax))
    Uspd=windSpd(uwnd,vwnd).mean(dim=['lev','lon'])
    # Buoyancy Frequency
    print('Calculating Zonal Mean Upstream N...')
    t0=ts.sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax))
    t1=ta[:,n_p2idx,:,:].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax))
    z0=topo.sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)) # get surface height from etopo dataset
    z1=z[:,n_p2idx,:,:].sel(lat=slice(ymin,ymax), lon=slice(xmin,xmax)) # height of atm at plevel
    g=np.full((len(t0.lat),len(t0.lon)), 9.81) # gravity
    theta=t1*((surface_plev/p1)**(2/7))
    dtdz=(t1-t0)/(z1-z0)
    dthetadz=((surface_plev/p1)**(2/7)) * (.00976 - np.abs(dtdz))
    N=np.sqrt((g/theta)*dthetadz).mean(dim='lon',skipna=True)
    
    ## Calculate Froude
    print('Calculating Froude...')
    Fr=froude(N,h,Uspd)
    print('Done.')
    return(Fr)

In [ ]:
### +++ LOAD DATA +++ ###

# set file info
dpath0='../topo_files' # top level data directory
files={ 'etopo':{}, 'ctrl':{}, 'max':{}, 'p85':{}, 'p80':{}, '1.5env':{}, '1env':{} }

files['etopo'] = f'{dpath0}/obs_data/froude.etopo05.nc'
files['ctrl'] = f'{dpath0}/output/froude.e2.1.topo.nc'
files['max'] = f'{dpath0}/output/froude.e2.1.topo.nc'
files['p85'] = f'{dpath0}/e2pt1.85p.ca-sa.zsurf.nc'
files['p80'] = f'{dpath0}/e2pt1.80p.ca-sa.zsurf.nc'
files['1.5env'] = f'{dpath0}/e2pt1.1pt5sEnv.ca-sa.zsurf.nc'
files['1env'] = f'{dpath0}/e2pt1.1sEnv.ca-sa.zsurf.nc'

files['land_mask'] = f'{dpath0}/Z2HX2fromZ1QX1N.BS1.nc' #e2pt1.ctrl.zsurf.nc'


# init dictionary
topo={ 'etopo':{}, 'ctrl':{}, 'max':{}, 'p85':{}, 'p80':{}, '1.5env':{}, '1env':{} }

for key in files.keys():
    
    # need to process the etopo data first
    if key in ['etopo']:
        etopo_full = xr.open_dataset(files[key]).ROSE
        # rename dimensions to match model data
        etopo_full = etopo_full.rename({'ETOPO05_Y':'lat', 'ETOPO05_X':'lon'}) 
        # switch from 0:360 to -180:180 lons
        nx = len(etopo_full.lon)
        lons = np.linspace(-180,180,nx)
        etopo_full['lon']=lons
        etopo_full=etopo_full.roll(lon=2160)
        topo[key] = etopo_full.where(etopo_full>0,np.nan) # remove bathymetry
        etopo_clip=etopo_full.sel(lat=slice(lat_min,lat_max), lon=slice(lon_min,lon_max))
    else:
        topo[key] = xr.open_dataset(files[key]).zatmo

# read in land mask from default surface boundary condition file
land_mask = xr.open_dataset(files[key]).fgrnd.sel(lat=slice(-55,25), lon=slice(-110,-60))

## Load MERRA-2 Climate Data
frvars=xr.open_dataset(f'{dpath0}/obs_data/frvars.climo.nc')

# MERRA-2 winds
l1=6 # 850 mb
dpath=f'{dpath0}/obs_data/merra2'
u=xr.open_dataset(f'{dpath}/merra2.U.1980-2022.monthly.nc').U[:,l1].groupby("time.month").mean().mean(dim='month')
v=xr.open_dataset(f'{dpath}/merra2.V.1980-2022.monthly.nc').V[:,l1].groupby("time.month").mean().mean(dim='month')

In [ ]:
### ----------------------------------------- ###
###    Calculate Model Froude with E2.1 BCs   ###
### ----------------------------------------- ###

modTopo_etopo_grid=modTopo.interp(lat=etopo_full.lat, lon=etopo_full.lon)

f1 = froude_from_upstream_vars(modTopo_etopo_grid,frvars.u,frvars.v,frvars.ts,frvars.ta,frvars.h,
                               p2wind=800, p2n=600,
                               oro_lat_bounds=[-55,-30], oro_lon_bounds=[-80,-60],
                               ups_lat_bounds=[-55,-30], ups_lon_bounds=[-95,-90])

f2 = froude_from_upstream_vars(modTopo_etopo_grid,frvars.u,frvars.v,frvars.ts,frvars.ta,frvars.h,
                               p2wind=800, p2n=600,
                               oro_lat_bounds=[-29.99,-10], oro_lon_bounds=[-80,-60],
                               ups_lat_bounds=[-29.99,-10], ups_lon_bounds=[-55,-50])

f3 = froude_from_upstream_vars(modTopo_etopo_grid,frvars.u,frvars.v,frvars.ts,frvars.ta,frvars.h,
                               p2wind=800, p2n=600,
                               oro_lat_bounds=[-9.99,11], oro_lon_bounds=[-90,-69],
                               ups_lat_bounds=[-9.99,11], ups_lon_bounds=[-65,-60])

f4 = froude_from_upstream_vars(modTopo_etopo_grid,frvars.u,frvars.v,frvars.ts,frvars.ta,frvars.h,
                               p2wind=800, p2n=600,
                               oro_lat_bounds=[11.01,25], oro_lon_bounds=[-110,-80],
                               ups_lat_bounds=[11.01,25], ups_lon_bounds=[-75,-70])

frGiss=xr.concat([f1,f2,f3,f4], dim='lat', coords='minimal')
frGiss=frGiss.transpose('source_id','month','lat','lon')
frGiss=frGiss.where(frGiss>0, np.nan)
frGiss.attrs['long_name']='Froude'
frGiss.attrs['wind_levels']='1000-800hPa'
frGiss.attrs['N_levels']='1000-600hPa'
del frGiss.attrs['units']

# make netcdf
month=frGiss.month
source_id=frGiss.source_id
lat=frGiss.lat
lon=frGiss.lon
frGiss=xr.Dataset(data_vars=dict(froude=(['source_id','month','lat','lon'], frGiss.data, frGiss.attrs),),
                  coords=dict(source_id=source_id,month=month,lat=lat,lon=lon,),)
frGiss.to_netcdf(f'{dpath0}/output/froude.e2.1.topo.nc', mode='w')

In [ ]:
### ----------------------------------------- ###
###  Calculate Observed Froude from ETOPO Obs ###
### ----------------------------------------- ###

f1 = froude_from_upstream_vars(etopo_full,frvars.u,frvars.v,frvars.ts,frvars.ta,frvars.h,
                               p2wind=800, p2n=600,
                               oro_lat_bounds=[-55,-30], oro_lon_bounds=[-80,-60],
                               ups_lat_bounds=[-55,-30], ups_lon_bounds=[-95,-90])

f2 = froude_from_upstream_vars(etopo_full,frvars.u,frvars.v,frvars.ts,frvars.ta,frvars.h,
                               p2wind=800, p2n=600,
                               oro_lat_bounds=[-29.99,-10], oro_lon_bounds=[-80,-60],
                               ups_lat_bounds=[-29.99,-10], ups_lon_bounds=[-55,-50])

f3 = froude_from_upstream_vars(etopo_full,frvars.u,frvars.v,frvars.ts,frvars.ta,frvars.h,
                               p2wind=800, p2n=600,
                               oro_lat_bounds=[-9.99,11], oro_lon_bounds=[-90,-69],
                               ups_lat_bounds=[-9.99,11], ups_lon_bounds=[-65,-60])

f4 = froude_from_upstream_vars(etopo_full,frvars.u,frvars.v,frvars.ts,frvars.ta,frvars.h,
                               p2wind=800, p2n=600,
                               oro_lat_bounds=[11.01,25], oro_lon_bounds=[-110,-80],
                               ups_lat_bounds=[11.01,25], ups_lon_bounds=[-75,-70])

frObs=xr.concat([f1,f2,f3,f4], dim='lat', coords='minimal')
frObs=frObs.where(frObs>0, np.nan)
frObs.attrs['long_name']='Froude'
frObs.attrs['wind_levels']='1000-800hPa'
frObs.attrs['N_levels']='1000-600hPa'
del frObs.attrs['units']

# make netcdf
month=frObs.month
lat=frObs.lat
lon=frObs.lon
frObs=xr.Dataset(data_vars=dict(froude=(['month','lat','lon'], frObs.data, frObs.attrs),),
                  coords=dict(month=month,lat=lat,lon=lon,),)
frObs.to_netcdf(f'{dpath0}/obs_data/froude.etopo05.nc', mode='w')

In [ ]:
## regrid model froude values to E2.1 grid
frGiss_lowres=frGiss.interp(lat=ocn_mask.lat, lon=ocn_mask.lon).sel(lat=slice(-55,25), lon=slice(-110,-60))

# keep only Froude values above critical value (>1)
ones=xr.ones_like(frGiss_lowres)
frGiss_idx=frGiss_lowres.where(frGiss_lowres>=1, np.nan)

# mask ocean cells
frGiss_idx=frGiss_idx.where(ocn_mask>0.25,np.nan)
frGiss_lowres=frGiss_lowres.where(ocn_mask>0.25,np.nan)

## Index Obs Froude
# keep only Froude values above critical value (>1)
ones=xr.ones_like(frObs)
frObs_idx=frObs.where(frObs>=1, np.nan)

In [ ]:
# -------------------- #
#       SETTINGS       #
# -------------------- #
text_kw={'color':'k', 'weight':'bold', 'size':'x-large', 'horizontalalignment':'left'}
arr_text_kw={'color':'blue', 'weight':'bold', 'size':'medium', 'horizontalalignment':'left', 'verticalalignment':'bottom'}
arrow_kw={'color':'blue', 'linewidth':.01, 'shrink':0.0}
# cmap specs
tcmap=cmo.deep
newcolors=tcmap(np.linspace(0.1, 1, 200))
white=np.repeat(np.array([[1, 1, 1, 1],[1, 1, 1, 1]]),25, axis=0)
cmap=ListedColormap(np.vstack((white, newcolors)))
vmin=0
vmax=5
# vector specs
skip_n=5
w=0.003
scalef=35
key_length=5
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-110,-60,-55,25]
titles=[]

# -------------------- #
#          FIG         #
# -------------------- #
fig, ax = plt.subplots(nrows=1, ncols=21, figsize=(25,5),
                       layout='constrained',
                       subplot_kw={'projection':proj},
                       gridspec_kw={'width_ratios': [3, 0.5, 0.15, 3, 0.5, 0.15, 3, 0.5, 0.15, 3, 0.5, 0.15, 3, 0.5, 0.15, 3, 0.5, 0.15, 3, 0.5, 0.15]}) #

## ETOPO Froude
ax[0].text(-109.5,26,'a. ETOPO05',**text_kw,zorder=100)
ax[0].pcolormesh(frObs_idx.lon, frObs_idx.lat, frObs.mean(dim='month'), cmap=cmap, vmin=vmin, vmax=vmax, transform=trans)
q1=ax[0].quiver(u.lon[::skip_n], u.lat[::skip_n], u[::skip_n,::skip_n], v[::skip_n,::skip_n],
                 color='k', width=w, scale=scalef, scale_units='inches', units='height', transform=trans, zorder=10)
qk=ax[0].quiverkey(q1, .9, 1.025, key_length, rf'{key_length} m/s', labelcolor='k', labelpos='W')
ax[0].annotate('TE', xy=(-94.25,18), xytext=(-95.25,23), label='_Hidden', arrowprops=arrow_kw, **arr_text_kw, zorder=100)
ax[0].annotate('PG', xy=(-84,12.2), xytext=(-80,15.5), label='_Hidden', arrowprops=arrow_kw, **arr_text_kw, zorder=100)
ax[0].annotate('PN', xy=(-79,9.5), xytext=(-76,13.25), label='_Hidden', arrowprops=arrow_kw, **arr_text_kw, zorder=100)
ax[0].set(aspect='auto')
ax[0].coastlines(color='k')
ax[0].add_feature(cfeature.OCEAN, fc=(0.9, 0.9, 0.9), zorder=0)
ax[0].add_feature(cfeature.LAND, fc=(0.9, 0.9, 0.9), zorder=0)
ax[0].set_extent(map_bnds, crs=trans)
gl=ax[0].gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
gl.xlocator=mticker.FixedLocator([-100,-90,-80,-70])
gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
gl.xformatter=LONGITUDE_FORMATTER
gl.yformatter=LATITUDE_FORMATTER
gl.xlabel_style={'color': 'black', 'weight': 'normal'}
gl.ylabel_style={'color': 'black', 'weight': 'normal'}
# heatmap
temp=frObs.mean(dim='month', skipna=True).interp_like(frGiss_lowres).max(dim='lon')
vec=np.flip(np.array([temp]))
frObs_zm=vec.reshape(vec.shape[1], 1)
sns.heatmap(frObs_zm, vmin=vmin, vmax=vmax, cmap=cmap, yticklabels=False, xticklabels=False, square=True, cbar=False, ax=ax[1])
ax[1].set(aspect=0.55)

ax[2].set_visible(False)
ax[5].set_visible(False)
ax[8].set_visible(False)
ax[11].set_visible(False)
ax[14].set_visible(False)
ax[17].set_visible(False)
ax[20].set_visible(False)

for i in [0,1,2,3,4,5]:
    axn=[3,6,9,12,15,18] #[2,4,6,8,10,12]
    n=axn[i]
    letter=['b. ','c. ','d. ','e. ','f. ','g. ', 'h. ', 'i. ']
    ax[n].text(-109.5,26,letter[i]+str(frGiss[i].source_id.values),**text_kw,zorder=100)
    
    ## plot Froude map
    ax[n].pcolormesh(frGiss_idx.lon, frGiss_idx.lat, frGiss_lowres[i].mean(dim='month'), cmap=cmap, vmin=vmin, vmax=vmax, transform=trans)
    ax[n].set(aspect='auto')
    ax[n].coastlines(color='k')
    ax[n].add_feature(cfeature.OCEAN, fc=(0.9, 0.9, 0.9), zorder=0)
    ax[n].add_feature(cfeature.LAND, fc=(0.9, 0.9, 0.9), zorder=0)
    ax[n].set_extent(map_bnds, crs=trans)
    gl=ax[n].gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.xlocator=mticker.FixedLocator([-100,-90,-80,-70])
    gl.bottom_labels=True; gl.left_labels=False; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal'}
    gl.ylabel_style={'color': 'black', 'weight': 'normal'}
    
    ## plot max Froude heat map
    vec=np.flip(np.array([frGiss_lowres[i].mean(dim='month').max(dim='lon')]))
    frGiss_zm=vec.reshape(vec.shape[1], 1)
    sns.heatmap(frGiss_zm, vmin=vmin, vmax=vmax, cmap=cmap, yticklabels=False, xticklabels=False, square=True, cbar=False, ax=ax[n+1])
    ax[n+1].set(aspect=.55)

    
# add colorbar
levels=np.linspace(vmin, vmax, 11)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
cf=mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
cax=fig.add_axes([.245, -0.075, 0.5, 0.05])
cbar=fig.colorbar(cf, orientation='horizontal', extend='max', cax=cax)
cbar.set_label('$F$', x=-0.025, labelpad=-32, rotation=0, size='xx-large', fontweight='bold', ha='center')
cbar.ax.tick_params(labelsize='x-large')
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')
